#Descripción del sistema RAG
Mi sistema RAG permite que docentes de IPC puedan chequear información, parafrasear ideas, buscar información determinada, comparar autores
consultando información del manual de lógica, filosofía e historia de la ciencia titulado *Desenredando la ciencia* mediante búsqueda semántica con ChromaDB y generación con Gemini..


#Configuración del entorno

In [ ]:
# Instalamos las librerías necesarias para nuestro sistema RAG
# LangChain: la librería más popular para construir sistemas RAG de forma simple
# ChromaDB: nuestra base de datos vectorial para guardar los documentos
# Google GenerativeAI: para conectar con Gemini (solo para generación final)
# sentence-transformers: para embeddings locales multilenguaje
!pip install langchain langchain-google-genai langchain-chroma chromadb sentence-transformers -q

print("Todas las librerías instaladas correctamente")
print("IMPORTANTE: Solo se usará API de Gemini para generación final de respuestas")

In [ ]:
# Importamos todas las herramientas que vamos a necesitar
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from chromadb.utils import embedding_functions

# Configuración para mostrar mejor los resultados
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas exitosamente")
print("Usando embeddings locales para reducir uso de API")

Librerías cargadas exitosamente
Usando embeddings locales para reducir uso de API


#Configuración de la API

In [ ]:
# Detectamos si estamos en Google Colab o en un entorno local
try:
    import google.colab
    from google.colab import userdata
    IN_COLAB = True
    print("🔍 Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("🔍 Entorno detectado: Local")

# Obtenemos la clave API según el entorno
if IN_COLAB:
    # En Colab: usar los secretos de Colab (más seguro)
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
        print("✅ Clave API cargada desde secretos de Colab")
    except Exception as e:
        print("❌ No se encontró GEMINI_API_KEY en los secretos de Colab")
        print("   Ve a la barra lateral izquierda > 🔑 Secretos > Agregar GOOGLE_API_KEY")
        GEMINIE_API_KEY = input("Pega tu clave API de Google aquí: ")
else:
    # En local: usar variable de entorno
    GEMINIE_API_KEY = os.getenv('GEMINI_API_KEY')
    if not GEMINI_API_KEY:
        print("❌ No se encontró GEMINI_API_KEY en las variables de entorno")
        print("   Opción 1: Agrega GEMINI_API_KEY a tu archivo .env")
        print("   Opción 2: Ejecuta: export GEMINI_API_KEY=tu_clave_aqui")
        GEMINI_API_KEY = input("Pega tu clave API de Google aquí: ")
    else:
        print("✅ Clave API cargada desde variables de entorno")

# Configuramos la variable de entorno para que LangChain la use
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print("🚀 Configuración de Gemini completada")

🔍 Entorno detectado: Google Colab
✅ Clave API cargada desde secretos de Colab
🚀 Configuración de Gemini completada


#Carga de Documentos PDF

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado correctamente en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montado correctamente en /content/drive


In [ ]:
# Instalamos la librería para trabajar con PDFs
# PyPDF2 es la librería más popular para extraer texto de archivos PDF
!pip install pypdf -q

print("PyPDF instalado correctamente")

PyPDF instalado correctamente


In [ ]:
!pip install -U langchain-community -q
print("langchain-community instalado correctamente")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.5 which is incompatible.
langchain 0.3.27 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.0.0 which is incompatible.
langchain-community instalado correctamente


In [ ]:
# Cargamos un PDF usando PyPDFLoader de LangChain
from langchain.document_loaders import PyPDFLoader

# Especificamos la ruta del archivo PDF
# En un caso real, aquí pondrías la ruta a tu propio documento
loader = PyPDFLoader("/content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf")

# Cargamos todas las páginas del PDF
pages = loader.load()

print(f"PDF cargado exitosamente: {len(pages)} páginas")
print("Cada página se convierte en un objeto Document independiente")

PDF cargado exitosamente: 410 páginas
Cada página se convierte en un objeto Document independiente


#Estructura de un Document en LangChain

In [ ]:
# Verificamos cuántas páginas se cargaron
len(pages)

410

In [ ]:
# Examinamos la primera página como ejemplo
page = pages[0]
print("Primera página seleccionada para análisis")

Primera página seleccionada para análisis


In [ ]:
# Mostramos los primeros 500 caracteres del contenido de la página
print("CONTENIDO DE LA PRIMERA PÁGINA (primeros 500 caracteres):")
print("=" * 60)
print(page.page_content[0:500])
print("=" * 60)
print("(Contenido truncado para visualización)")

CONTENIDO DE LA PRIMERA PÁGINA (primeros 500 caracteres):
Colección UBA XXI 
Desenredando la ciencia 
IPC_2021_correcc_10_1_2022_v02   1IPC_2021_correcc_10_1_2022_v02   1 18/1/2022   11:54:4018/1/2022   11:54:40
(Contenido truncado para visualización)


In [ ]:
# Examinamos los metadatos de la página
print("METADATOS DE LA PÁGINA:")
print("=" * 30)
print(page.metadata)
print("=" * 30)
print("Los metadatos incluyen información como número de página y archivo fuente")

METADATOS DE LA PÁGINA:
{'producer': 'Adobe PDF Library 16.0.3', 'creator': 'Adobe InDesign 17.0 (Windows)', 'creationdate': '2022-01-10T17:51:03-03:00', 'moddate': '2022-02-23T11:38:08-03:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf', 'total_pages': 410, 'page': 0, 'page_label': '1'}
Los metadatos incluyen información como número de página y archivo fuente


#Paso 1: Dividir los Documentos en Fragmentos

In [ ]:
# El "Text Splitter" es como un bibliotecario que divide documentos grandes
# en secciones manejables, manteniendo el contexto

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Cada fragmento tendrá máximo 500 caracteres
    chunk_overlap=50,      # 50 caracteres se superponen entre fragmentos para mantener contexto
    separators=["\n\n", "\n", ".", " "]  # Divide preferentemente por párrafos, luego oraciones
)

# Convertimos nuestros documentos al formato que entiende LangChain
documentos_langchain = []

# Iterate over the loaded pages instead of the loader object
for doc in pages:
    # Each page is already a "Document" object with content and metadata
    # We can directly append it to the list
    documentos_langchain.append(doc)


# Dividimos todos los documentos en fragmentos más pequeños
fragmentos = text_splitter.split_documents(documentos_langchain)

print(f"📝 Documentos originales: {len(documentos_langchain)}")
print(f"🔪 Fragmentos creados: {len(fragmentos)}")
print(f"\n📋 Ejemplo de fragmento:")
print(f"Título: {fragmentos[0].metadata['source']}") # Use 'source' metadata
print(f"Contenido: {fragmentos[0].page_content[:200]}...")

📝 Documentos originales: 410
🔪 Fragmentos creados: 2169

📋 Ejemplo de fragmento:
Título: /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
Contenido: Colección UBA XXI 
Desenredando la ciencia 
IPC_2021_correcc_10_1_2022_v02   1IPC_2021_correcc_10_1_2022_v02   1 18/1/2022   11:54:4018/1/2022   11:54:40...


In [ ]:
print(f"Título: {fragmentos[2].metadata['source']}")
print(f"Contenido: {fragmentos[2].page_content[:200]}...")

Título: /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
Contenido: Colección UBA XXI
UBA XXI es el programa de la Universidad de Buenos Aires que ofrece 
la posibilidad de cursar a distancia las materias del Ciclo Básico Común 
de todas las carreras. Se trata de una ...


#Paso 2: Crear la Base de Conocimiento Vectorial

In [ ]:
# Los "embeddings" convierten texto en vectores numéricos que representan el significado
# Usamos un modelo local multilenguaje que funciona excelente con español técnico
embeddings = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-large"  # Modelo multilenguaje optimizado para español
)

print("Modelo de embeddings local configurado (multilingual-e5-large)")
print("Ventaja: No consume cuota de API, solo procesamiento local")

Modelo de embeddings local configurado (multilingual-e5-large)
Ventaja: No consume cuota de API, solo procesamiento local


In [ ]:
!pip install langchain_community -q

In [ ]:
# ChromaDB será nuestra "biblioteca inteligente" donde guardamos los vectores
# Es como un bibliotecario que puede encontrar libros por su tema, no solo por título
from langchain_community.embeddings import SentenceTransformerEmbeddings

embeddings = SentenceTransformerEmbeddings(model_name="intfloat/multilingual-e5-large")

vectorstore = Chroma.from_documents(
    documents=fragmentos,           # Los fragmentos de nuestros documentos
    embedding=embeddings,           # El modelo que convierte texto en vectores
    collection_name="documentos_empresa",  # Nombre de nuestra colección
    persist_directory="./chroma_db"  # Donde se guardan los datos (opcional)
)

print(f"Base de conocimiento vectorial creada con {len(fragmentos)} fragmentos")
print("El sistema ya puede buscar información por significado, no solo por palabras exactas")
print("Los embeddings se procesan localmente sin consumir cuota de Gemini")

Base de conocimiento vectorial creada con 2169 fragmentos
El sistema ya puede buscar información por significado, no solo por palabras exactas
Los embeddings se procesan localmente sin consumir cuota de Gemini


In [ ]:
# Configuramos el modelo Gemini que generará las respuestas finales
# NOTA: Solo este componente consume cuota de API, los embeddings son locales
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",    # Modelo rápido y eficiente de Gemini
    temperature=0.1,             # Baja creatividad = respuestas más precisas y consistentes
    google_api_key=GEMINI_API_KEY
)

print("Modelo Gemini configurado")
print("   Modelo: models/gemini-2.5-flash (rápido y preciso)")
print("   Temperatura: 0.1 (respuestas consistentes y factuales)")
print("   IMPORTANTE: Solo la generación final usa API de Gemini")

Modelo Gemini configurado
   Modelo: models/gemini-2.5-flash (rápido y preciso)
   Temperatura: 0.1 (respuestas consistentes y factuales)
   IMPORTANTE: Solo la generación final usa API de Gemini


#Paso 3: Crear el Template de Respuesta

In [ ]:
# El prompt template es como las instrucciones que le damos a un asistente
# Le decimos exactamente como debe comportarse y que formato usar

template_respuesta = """
Sos un asistente experto en filosofía de las ciecias, lógica e historia de las ciencias.
Tu trabajo es responder preguntas basandote UNICAMENTE en la informacion
proporcionada en el manual de UBA XXI.

INSTRUCCIONES IMPORTANTES:
1. Solo usa informacion que aparece explicitamente en los documentos
2. Si no encontras la informacion especifica, decilo claramente
3. Cita el documento o seccion cuando sea posible
4. Se preciso con nombres, fechas y datos
5. Usa un tono amigable pero informado

CONTEXTO DE LOS DOCUMENTOS:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:
"""

# Creamos el prompt personalizado usando nuestro template
prompt = PromptTemplate(
    template=template_respuesta,
    input_variables=["context", "question"]
)

print("Template de respuesta configurado")
print("El asistente seguira instrucciones especificas para dar respuestas precisas")

Template de respuesta configurado
El asistente seguira instrucciones especificas para dar respuestas precisas


#Paso 4: Ensamblar el Sistema RAG Completo

In [ ]:
# El "RetrievalQA" es el corazón de nuestro sistema RAG
# Conecta la búsqueda (Retrieval) con la generación (QA = Question Answering)

sistema_rag = RetrievalQA.from_chain_type(
    llm=llm,                              # Nuestro modelo Gemini
    chain_type="stuff",                   # Estrategia: "meter" toda la info relevante en el prompt
    retriever=vectorstore.as_retriever(   # Configuración del buscador
        search_kwargs={"k": 3}            # Buscar los 3 fragmentos más relevantes
    ),
    chain_type_kwargs={                   # Configuraciones adicionales
        "prompt": prompt,                 # Nuestras instrucciones personalizadas
        "verbose": False                  # No mostrar pasos internos (para mantenerlo limpio)
    },
    return_source_documents=True          # Devolver también los documentos fuente
)

print("Sistema RAG completamente configurado")
print("\nFlujo de trabajo del sistema:")
print("   1. Usuario hace una pregunta")
print("   2. El sistema busca los 3 fragmentos más relevantes (LOCAL)")
print("   3. Gemini lee esos fragmentos y genera una respuesta (API)")
print("   4. Se devuelve la respuesta + documentos fuente")
print("\nVentaja: 70-80% menos uso de API de Gemini")
print("Listo para responder preguntas!")

Sistema RAG completamente configurado

Flujo de trabajo del sistema:
   1. Usuario hace una pregunta
   2. El sistema busca los 3 fragmentos más relevantes (LOCAL)
   3. Gemini lee esos fragmentos y genera una respuesta (API)
   4. Se devuelve la respuesta + documentos fuente

Ventaja: 70-80% menos uso de API de Gemini
Listo para responder preguntas!


#Probando Nuestro Sistema RAG

In [ ]:
# Función auxiliar para mostrar respuestas de forma clara y educativa
def hacer_pregunta(pregunta, mostrar_fuentes=True):
    """
    Función que procesa una pregunta y muestra la respuesta de forma educativa

    Args:
        pregunta (str): La pregunta que queremos hacer al sistema
        mostrar_fuentes (bool): Si mostrar o no los documentos fuente
    """
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {pregunta}")
    print(f"{'='*60}")

    # Enviamos la pregunta al sistema RAG
    resultado = sistema_rag({"query": pregunta})

    # Mostramos la respuesta generada por Gemini
    print(f"\nRESPUESTA DEL SISTEMA:")
    print(resultado["result"])

    # Opcionalmente mostramos las fuentes consultadas
    if mostrar_fuentes and resultado["source_documents"]:
        print(f"\nDOCUMENTOS CONSULTADOS:")
        for i, doc in enumerate(resultado["source_documents"], 1):
            print(f"   {i}. {doc.metadata['source']}")
            print(f"      Fragmento: {doc.page_content[:100]}...")

    return resultado

print("Función de prueba lista")
print("Ahora podemos hacer preguntas específicas sobre nuestros manual de lógica, filosofía e historia de las ciencia")

Función de prueba lista
Ahora podemos hacer preguntas específicas sobre nuestros manual de lógica, filosofía e historia de las ciencia


#Tipos de preguntas posibles

In [ ]:
#Ejemplo sobre Kuhn
# Preguntamos sobre la concepción de la ciencia de Kuhn
resultado1 = hacer_pregunta("¿Cuando un paradigma logra constituirse en ciencia normal, cuál es el trabajo de los científicos?")


PREGUNTA: ¿Cuando un paradigma logra constituirse en ciencia normal, cuál es el trabajo de los científicos?

RESPUESTA DEL SISTEMA:
¡Hola! Con gusto te explico el trabajo de los científicos cuando un paradigma logra constituirse en ciencia normal, basándome en la información proporcionada.

Cuando un paradigma logra constituirse en ciencia normal, el trabajo de los científicos se aboca a la **resolución de enigmas**. Este es el rasgo distintivo y el tipo de práctica que caracteriza a la ciencia normal.

La ciencia normal es el período de trabajo científico que ocurre mientras se mantiene el consenso en la comunidad científica en torno a la vigencia de un cierto paradigma.

(Referencia: "Ahora bien, antes de continuar con la descripción de la ciencia normal...", "Se trata del período de trabajo científico...", "Su rasgo distintivo, es decir, lo que caracteriza a la ciencia normal es su tipo de práctica, a saber, ella se aboca a la resolución de enigmas.")

DOCUMENTOS CONSULTADOS:
   1.

In [ ]:
#Ejemplo sobre Popper y el falsacionismo
# Preguntamos sobre el criterio de demarcación científica
resultado2 = hacer_pregunta("¿Cuáles son las condiciones para que una teoría pueda considerarse científica según Popper?")


PREGUNTA: ¿Cuáles son las condiciones para que una teoría pueda considerarse científica según Popper?

RESPUESTA DEL SISTEMA:
¡Hola! Con gusto te ayudo con tu pregunta basándome en la información del manual de UBA XXI.

Según el criterio popperiano, para que las hipótesis sean científicas, deben poder ser refutadas en una contrastación experimental. Aunque la idea es que no lo sean efectivamente, la posibilidad de ser refutadas es la condición clave.

(Fuente: El criterio popperiano, párrafo 1)

DOCUMENTOS CONSULTADOS:
   1. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: las investigaciones científicas. 
En su tarea de revisión crítica, Popper analizó una de esas estrat...
   2. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: las investigaciones científicas. 
En su tarea de revisión crítica, Popper analizó una de esas estrat...
   3. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando 

In [ ]:
#Ejemplo sobre lógica
# Preguntamos sobre la noción de validez
resultado3 = hacer_pregunta("¿Cuando un argumento es válido?")


PREGUNTA: ¿Cuando un argumento es válido?

RESPUESTA DEL SISTEMA:
¡Hola! Con gusto te explico cuándo un argumento es válido, basándome en la información que me proporcionaste del manual de UBA XXI.

Un argumento es **válido** cuando "garantiza que, si las premisas son verdaderas, la conclusión también lo será, pero no garantiza que sus premisas sean efectivamente verdaderas."

De modo equivalente, en el caso de los argumentos válidos, "resulta imposible que las premisas sean verdaderas y que la conclusión sea falsa."

Espero que esta explicación te sea de gran ayuda. ¡Cualquier otra consulta, no dudes en preguntar!

DOCUMENTOS CONSULTADOS:
   1. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: La validez de un argumento garantiza que, si las premisas son verdade-
ras, la conclusión también lo...
   2. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: La validez de un argumento garantiza que,

In [ ]:
#Ejemplo sobre historia de la ciencia
# Preguntamos sobre la diferencia entre las concepciones de Darwin y Lamarck
resultado4 = hacer_pregunta("¿Me hacés una compración entre las concepciones de Darwin y Lamarck?")


PREGUNTA: ¿Me hacés una compración entre las concepciones de Darwin y Lamarck?

RESPUESTA DEL SISTEMA:
¡Hola! Con gusto te ayudo a comparar las concepciones de Darwin y Lamarck basándome UNICAMENTE en la información proporcionada en el manual de UBA XXI.

Aquí tienes una comparación entre ambos, según el texto:

*   **La idea de Evolución:**
    *   **Lamarck:** Es un precedente de Darwin y su tesis central es el **evolucionismo**, que postula que "las especies se transforman" (Sección "Lamarck y el evolucionismo"). Esta idea se opone al fijismo.
    *   **Darwin:** Si bien la idea de evolución ya era familiar antes de la aparición de su teoría, Darwin también se asocia con esta idea. Su teoría de la selección natural explica cómo las distintas especies emergen unas de las otras (CAPÍTULO 8. LA REVOLUCIÓN DARWINIANA).

*   **El Mecanismo de Transformación de las Especies:**
    *   **Lamarck:** El manual menciona que Lamarck propuso que "las especies se transforman". Sin embargo, la i

#Pregunta que NO está en los documentos

In [ ]:
# Probamos qué pasa cuando preguntamos algo que no está en nuestros documentos
resultado6 = hacer_pregunta("¿Cuál es el procedimiento para solicitar una computadora nueva?")


PREGUNTA: ¿Cuál es el procedimiento para solicitar una computadora nueva?

RESPUESTA DEL SISTEMA:
¡Hola!

He revisado cuidadosamente los documentos que me proporcionaste, pero no encuentro ninguna información específica sobre el procedimiento para solicitar una computadora nueva.

Los textos hablan sobre el desarrollo de técnicas en plantas de energía nuclear, criterios tecnológicos, patentes y la definición de un procedimiento algorítmico, pero no mencionan nada relacionado con la solicitud de equipos informáticos.

Por lo tanto, no puedo responder a tu pregunta basándome únicamente en la información disponible aquí.

DOCUMENTOS CONSULTADOS:
   1. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: energía puede desarrollar una nueva técnica de organización de plan-
tas de energía nuclear. Usualme...
   2. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: energía puede desarrollar una nueva té

In [ ]:
# Probamos qué pasa cuando preguntamos algo que no está en nuestros documentos
resultado5 = hacer_pregunta("¿Cuál es la concepción de la ciencia de Latour?")


PREGUNTA: ¿Cuál es la concepción de la ciencia de Latour?

RESPUESTA DEL SISTEMA:
¡Hola!

Según la información proporcionada en el manual de UBA XXI, no se encuentra ninguna mención explícita sobre la concepción de la ciencia de Latour.

El texto se enfoca en la concepción clásica de las teorías empíricas como sistemas axiomáticos interpretados y en la concepción positivista del progreso científico, pero no aborda las ideas de Latour.

DOCUMENTOS CONSULTADOS:
   1. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: de la ciencia se orienta al análisis lógico de las teorías científicas. Se-
gún la concepción clásic...
   2. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: de la ciencia se orienta al análisis lógico de las teorías científicas. Se-
gún la concepción clásic...
   3. /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Fragmento: cación de la cienc

#Análisis del Sistema: ¿Cómo funciona internamente?

In [ ]:
# Función para mostrar el proceso interno paso a paso
def analizar_proceso(pregunta):
    """
    Función educativa que muestra todos los pasos internos del proceso RAG
    """
    print(f"\nANALISIS PASO A PASO")
    print(f"Pregunta: {pregunta}")
    print("\n" + "="*50)

    # PASO 1: Búsqueda de documentos relevantes
    print("PASO 1: Búsqueda Vectorial (LOCAL - sin usar API)")
    documentos_relevantes = vectorstore.similarity_search(pregunta, k=3)

    print(f"   Se encontraron {len(documentos_relevantes)} fragmentos relevantes:")
    for i, doc in enumerate(documentos_relevantes, 1):
        print(f"   {i}. De: {doc.metadata['source']}")
        print(f"      Contenido: {doc.page_content[:150]}...\n")

    # PASO 2: Construcción del contexto
    print("PASO 2: Construcción del Contexto (LOCAL)")
    contexto = "\n\n".join([doc.page_content for doc in documentos_relevantes])
    print(f"   Se combinaron {len(documentos_relevantes)} fragmentos en un contexto de {len(contexto)} caracteres")

    # PASO 3: Generación del prompt final
    print("\nPASO 3: Prompt Final para Gemini (LOCAL)")
    prompt_final = prompt.format(context=contexto, question=pregunta)
    print("   Longitud del prompt:", len(prompt_final), "caracteres")
    print("   Primeras líneas del prompt:")
    print("   " + "\n   ".join(prompt_final.split("\n")[:8]))

    # PASO 4: Respuesta del modelo
    print("\nPASO 4: Generación de Respuesta (USA API DE GEMINI)")
    resultado = sistema_rag({"query": pregunta})
    print(f"   Respuesta generada: {len(resultado['result'])} caracteres")
    print(f"   Respuesta: {resultado['result']}")

    return resultado

# Analicemos una pregunta específica
analisis = analizar_proceso("¿Cuándo se puede trabajar remotamente?")


ANALISIS PASO A PASO
Pregunta: ¿Cuándo se puede trabajar remotamente?

PASO 1: Búsqueda Vectorial (LOCAL - sin usar API)
   Se encontraron 3 fragmentos relevantes:
   1. De: /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Contenido: lentos para una situación de emergencia. 
Ante esto, los filósofos Remco Heesen y Liam Kofi Bright (2021) sostie-
nen que los costos del sistema de re...

   2. De: /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Contenido: lentos para una situación de emergencia. 
Ante esto, los filósofos Remco Heesen y Liam Kofi Bright (2021) sostie-
nen que los costos del sistema de re...

   3. De: /content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf
      Contenido: están actuando en calidad de representantes de sus empleadores. No 
siempre es fácil hacer esta distinción. Muchos de los médicos que tra-
bajaban par...

PASO 2: Construcción del Contexto (LOCAL)
   Se c

#Interfaz con Steamlit

In [ ]:
!pip install -q streamlit
print("Streamlit instalado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 125.9 MB/s eta 0:00:00
Streamlit instalado correctamente.


In [ ]:
%%writefile app.py

# Importaciones necesarias para el script de Streamlit
import streamlit as st
import os

# --- Lógica del Sistema RAG (Debe estar autocontenida aquí) ---
# Se necesita para que 'qa_chain' exista al correr el script

# 1. Configuración de Librerías y Rutas
# Asume que ya se instalaron:
# langchain langchain-google-genai langchain-chroma chromadb sentence-transformers pypdf langchain-community

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_chroma import Chroma
    from langchain.chains import RetrievalQA
    from langchain.schema import Document
    from langchain.prompts import PromptTemplate
    from chromadb.utils import embedding_functions
    from langchain_community.embeddings import SentenceTransformerEmbeddings
    from langchain.document_loaders import PyPDFLoader
except ImportError:
    st.error("Por favor, asegúrate de que todas las librerías estén instaladas en tu entorno.")
    st.stop()


# 2. Configuración de la API (Tomando de la variable de entorno ya configurada)
GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
if not GOOGLE_API_KEY:
    st.error("No se encontró la GEMINI_API_KEY. Por favor, asegúrate de haberla configurado en el notebook antes de correr Streamlit.")
    st.stop()

# Ruta fija del PDF (Asegúrate de que Google Drive esté montado previamente en el notebook)
PDF_PATH = "/content/drive/MyDrive/IPC/Buacar (2022) Desenredando la ciencia - Completo.pdf"


@st.cache_resource
def setup_rag_system():
    """Inicializa y configura todo el sistema RAG."""
    try:
        # A. Carga de Documentos
        loader = PyPDFLoader(PDF_PATH)
        pages = loader.load()

        # B. División de Fragmentos
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ".", " "]
        )
        fragmentos = text_splitter.split_documents(pages)

        # C. Creación del Vectorstore (Embeddings Locales)
        embeddings = SentenceTransformerEmbeddings(model_name="intfloat/multilingual-e5-large")
        vectorstore = Chroma.from_documents(
            documents=fragmentos,
            embedding=embeddings,
            collection_name="documentos_empresa",
            # No persistimos el directorio en este caso, se carga en memoria para Streamlit
        )

        # D. Configuración del Modelo y Prompt
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0.1,
            google_api_key=GEMINI_API_KEY
        )

        template_respuesta = """
        Sos un asistente experto en filosofía de las ciecias, lógica e historia de las ciencias.
        Tu trabajo es responder preguntas basandote UNICAMENTE en la informacion
        proporcionada en el manual de UBA XXI.

        INSTRUCCIONES IMPORTANTES:
        1. Solo usa informacion que aparece explicitamente en los documentos
        2. Si no encontras la informacion especifica, decilo claramente
        3. Cita el documento o seccion cuando sea posible
        4. Se preciso con nombres, fechas y datos
        5. Usa un tono amigable pero informado

        CONTEXTO DE LOS DOCUMENTOS:
        {context}

        PREGUNTA DEL USUARIO:
        {question}

        RESPUESTA:
        """
        prompt = PromptTemplate(
            template=template_respuesta,
            input_variables=["context", "question"]
        )

        # E. Ensamblaje del Sistema RAG
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
            chain_type_kwargs={"prompt": prompt, "verbose": False},
            return_source_documents=True
        )
        return qa_chain

    except Exception as e:
        st.error(f"Error al configurar el sistema RAG: {e}")
        st.info("Verifica la ruta del PDF y la clave API.")
        st.stop()

# Inicializamos el sistema RAG una sola vez usando caché de Streamlit
qa_chain = setup_rag_system()

# --- Interfaz de Streamlit (Se mantiene similar) ---

st.set_page_config(page_title="Mi Sistema RAG", page_icon="🔍")

st.title("Sistema RAG: 🧠 Pensamiento Científico (IPC)")
st.markdown("Consultá información de **lógica, filosofía e historia de la ciencia** basándote en el documento 'Desenredando la ciencia'.")

# Sidebar con información
with st.sidebar:
    st.header("Información")
    st.write("Corpus: Desenredando la ciencia (UBA XXI)")
    st.write("Modelo: gemini-2.5-flash")
    st.markdown("---")
    st.subheader("Ejemplos de consultas:")
    st.write("1. ¿Cuándo un argumento es válido?")
    st.write("2. Cuando un paradigma logra constituirse en ciencia normal, ¿cuál es el trabajo de los científicos? ")

# Input principal
consulta = st.text_input(
    "Escribí tu consulta:",
    placeholder="Ejemplo: ¿Cuál es el criterio de demarcación de Popper?"
)

if st.button("Consultar", type="primary"):
    if not consulta:
        st.warning("Por favor, escribí una consulta.")
    else:
        with st.spinner("Buscando información relevante y generando respuesta con Gemini..."):
            try:
                # La variable es qa_chain ahora
                resultado = qa_chain({"query": consulta})

                st.success("Consulta completada")

                # Respuesta
                st.subheader("Respuesta:")
                st.write(resultado["result"])

                # Fuentes
                st.subheader("Fuentes consultadas:")
                for i, doc in enumerate(resultado["source_documents"], 1):
                    # Usamos .get() para evitar errores si falta la metadata 'source'
                    source_name = doc.metadata.get('source', 'Desconocido').split('/')[-1]
                    with st.expander(f"Fuente {i}: {source_name}"):
                        st.write(doc.page_content) # Mostramos el contenido completo del fragmento
                        st.caption(f"Página: {doc.metadata.get('page', 'N/A')}")

            except Exception as e:
                st.error(f"Error al procesar la consulta: {str(e)}")
                st.info("Intentá reformular tu consulta o verifica la configuración del RAG en el notebook.")

Writing app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://honest-years-act.loca.lt
/root/.npm/_npx/75ac80b86e83d4a2/node_modules/localtunnel/bin/lt.js:81
    throw err;
    ^

Error: connection refused: localtunnel.me:15697 (check your firewall settings)
    at Socket.<anonymous> (/root/.npm/_npx/75ac80b86e83d4a2/node_modules/localtunnel/lib/TunnelCluster.js:52:11)
    at Socket.emit (node:events:524:28)
    at emitErrorNT (node:internal/streams/destroy:169:8)
    at emitErrorCloseNT (node:internal/streams/destroy:128:3)
    at process.processTicksAndRejections (node:internal/process/task_queues:82:21)

Node.js v20.19.0
⠙